In [1]:
import pandas as pd
import numpy as np
import os
import scipy.stats as st
import matplotlib.pyplot as plt
import sys
sys.path.append('../../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils

from statsmodels.stats.proportion import proportions_ztest


In [2]:
expected_mutations = pd.read_csv('data/GBM_expected_mutations.csv')[['sample','name','mutation_freq']]
expected_mutations.rename(columns={'mutation_freq': 'expected_frequency_from_bulk'}, inplace=True)

expected_mutations['HGVSc'] = expected_mutations['name'].str.split(' ').str[1]
expected_mutations['gene'] = expected_mutations['name'].str.split(' ').str[0]

all_expected_mutations = expected_mutations.copy()


In [3]:
merged_table_4plex_dcl = gf_utils.make_merge_table('output/unexpected_gapfills_GBM', label_control_column=False, lib='GBM')
merged_table_ref = gf_utils.make_merge_table('data/unexpected_gapfills_ref', label_control_column=False, lib='other') ## read in reference gapfills from other patients

In [4]:
merge_columns = ['gapfill','gapfill_from_transcriptome','gap_probe_sequence','gapfill_start','likelihood_given_wt_edit_dist','lhs_probe','rhs_probe']

# Merge only if the DataFrames are not empty
if not merged_table_4plex_dcl.empty:
    merged_table = merged_table_4plex_dcl.merge(merged_table_ref, on=merge_columns, how='outer')
else:
    merged_table = merged_table_ref.copy()

### manual fix for EGFR VIII wt control
merged_table.loc[(merged_table['lhs_probe'] == 'TACGGGGACACTTCTTCACGCAGGT') & (merged_table['rhs_probe'] == 'ACCAAAGCTGTATTTGCCCTCGGGG') & (merged_table['gapfill'] == merged_table['gapfill_from_transcriptome']),'lhs_probe'] = 'GAGCCGTGATCTGTCACCACATAAT'
merged_table.loc[(merged_table['lhs_probe'] == 'GAGCCGTGATCTGTCACCACATAAT') & (merged_table['rhs_probe'] == 'ACCAAAGCTGTATTTGCCCTCGGGG') & (merged_table['gapfill'] == merged_table['gapfill_from_transcriptome']),'rhs_probe'] = 'CTTTCTTTTCCTCCAGAGCCCGACT'

merged_table = gf_utils.sum_probe_counts(merged_table)


In [5]:
id_vars = list(merge_columns) + ['gap_probe_sequence'] + list(merged_table.columns[merged_table.columns.str.contains('control')])
value_vars = merged_table.columns.drop(id_vars)

samples = value_vars.str.split('probe_idx_').str[1].dropna().unique()
print('samples to parse:', samples)

dfs = []
for sample in samples:
    df_tmp = merged_table[id_vars].copy()
    for col in value_vars:
        if sample in col:
            df_tmp[col.replace('_' + sample, '')] = merged_table[col]
    df_tmp['sample'] = sample
    dfs.append(df_tmp)

merged_long = pd.concat(dfs, ignore_index=True)

merged_long['original_name'] = merged_long['name'].copy()

## drop rows where probe_idx is NaN
merged_long = merged_long.loc[merged_long['probe_idx'].notna()]

### add HGVSc based on observed gapfill
merged_long = gf_utils.name_variants_by_gapfill(merged_long, all_expected_mutations, merge_columns)

### manual fix for EGFR VIII
merged_long.loc[(merged_long['name'] == 'EGFR VIII') & (merged_long['gapfill'] == 'TAC'),'HGVSc'] = 'EGFR VIII'
merged_long.loc[(merged_long['name'] == 'EGFR c.866C>T') & (merged_long['HGVSc'] == 'EGFR wildtype'),'name'] = 'EGFR VIII' ## use this as control for EGFR VIII

## reorder columns for better interpretability
cols = ['name','HGVSc'] + merge_columns + [col for col in merged_long.columns if col not in (['name','HGVSc'] + merge_columns)]
merged_long = merged_long[cols]

merged_long['frequency'] = merged_long['count_of_this_gapfill'] / merged_long['count_of_this_probe']


samples to parse: Index(['BC002_GBM', 'BC004_exp2_other', 'BC001_exp2_other', 'BC004_other',
       'BC001_other', 'BC003_other', 'BC002_other', 'BC003_exp2_other'],
      dtype='object')
len merged long 23623
len merged long after merge 23623


In [6]:
merged_long = gf_utils.get_likelihood_given_edit_distance(merged_long)
merged_long = gf_utils.get_likelihood_given_control_sample(merged_long)
merged_long = gf_utils.get_likelihood_given_all_other_samples(merged_long)

merged_long['signed_log_likelihood_given_wt_control'] = np.log10(merged_long['likelihood_given_wt_control'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_control']
merged_long['signed_log_likelihood_given_other_samples'] = np.log10(merged_long['likelihood_given_other_samples'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_others']
merged_long['signed_log_likelihood_given_wt_edit_dist'] = np.log10(merged_long['likelihood_observed_proportion_given_edit_dist'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_edit_dist']

In [7]:
if 'expected_frequency_from_bulk' not in merged_long.columns:
    merged_long = merged_long.merge(expected_mutations.drop(columns=['HGVSc']).rename(columns={'name': 'HGVSc'}), on=['sample','HGVSc'], how='left')

In [8]:
mutated_df = merged_long.loc[~(merged_long['HGVSc'].fillna('').str.contains('wildtype'))].copy()


In [9]:
min_count = 0
min_freq = 0
min_log_likelihood_expected = -3
prior = 25 + min_log_likelihood_expected ### to keep a likelihood of -25 on unexpected
min_log_likelihood = -prior + min_log_likelihood_expected

temp_mutated_df = mutated_df.copy()

### add a prior on expected mutations
for col in ['signed_log_likelihood_given_wt_control', 'signed_log_likelihood_given_other_samples', 'signed_log_likelihood_given_wt_edit_dist']:
    temp_mutated_df.loc[(temp_mutated_df['expected_frequency_from_bulk'].notna()) & (temp_mutated_df[col].notna()), col] -= prior


### tiered approach to define final set
feature_sets = {}
for sample in mutated_df['sample'].unique():
    temp = gf_utils.get_feature_set(temp_mutated_df, sample, likelihood_column = 'signed_log_likelihood_given_wt_control', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_control'
    feature_set = temp.copy()
    temp = gf_utils.get_feature_set(temp_mutated_df.loc[temp_mutated_df['signed_log_likelihood_given_wt_control'].isna()], sample, likelihood_column = 'signed_log_likelihood_given_other_samples', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_others'
    feature_set = pd.concat([feature_set, temp], ignore_index=True)
    temp = gf_utils.get_feature_set(temp_mutated_df.loc[(temp_mutated_df['signed_log_likelihood_given_wt_control'].isna()) & (temp_mutated_df['signed_log_likelihood_given_other_samples'].isna())], sample, likelihood_column = 'signed_log_likelihood_given_wt_edit_dist', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_edit_dist'
    feature_set = pd.concat([feature_set, temp], ignore_index=True)
    feature_sets[sample] = feature_set

all_feature_sets = pd.concat(feature_sets.values(), ignore_index=True)

for col in ['signed_log_likelihood_given_wt_control', 'signed_log_likelihood_given_other_samples', 'signed_log_likelihood_given_wt_edit_dist']:
    all_feature_sets.loc[all_feature_sets['expected_frequency_from_bulk'].notna(), col] += prior

### make feature set to save using frequency and count thresholds only for the mutations that were not expected
all_features_to_save = all_feature_sets.loc[(all_feature_sets['expected_frequency_from_bulk'].notna()) | ((all_feature_sets['count_of_this_gapfill'] > 50) & (all_feature_sets['frequency'] > 0.003))]


In [10]:
### remove loci that frequently appear as novel positives
n_positives = all_features_to_save.loc[all_features_to_save['expected_frequency_from_bulk'].isna()]['name'].value_counts()
to_keep = n_positives[n_positives == 1].index
all_features_to_save = all_features_to_save.loc[(all_features_to_save['expected_frequency_from_bulk'].notna()) | (all_features_to_save['name'].isin(to_keep))]

### filter further in regime of precision > 0.6
all_features_to_save = all_features_to_save.loc[(all_features_to_save['expected_frequency_from_bulk'].notna()) | ((all_features_to_save['signed_log_likelihood_given_wt_control'] < (-25)) & (all_features_to_save['signed_log_likelihood_given_other_samples'] < (-50)))].sort_values('frequency')


In [11]:
all_features_to_save.to_csv('feature_set_all.csv', index=False)